In [ ]:
import pandas as pd
from pathlib import Path

In [6]:

ruta_geo = Path("..") / "data" / "raw" / "MUNICIPIOS.csv"

geo = pd.read_csv(
    ruta_geo,
    sep=';',            # separador ;
    encoding='latin1',  # o cp1252 si hiciera falta
    decimal=','         # convierte  -2,5124  -> -2.5124
)

# Limpiar posible BOM en la primera columna
geo.columns = [c.lstrip('\ufeff') for c in geo.columns]

# Códigos de provincias andaluzas
cod_andalucia = ['04', '11', '14', '18', '21', '23', '29', '41']

# Asegurar que COD_PROV es string de 2 dígitos
geo['COD_PROV'] = geo['COD_PROV'].astype(str).str.zfill(2)

# Filtrar municipios andaluces
geo_andalucia = geo[geo['COD_PROV'].isin(cod_andalucia)].copy()

# Asegurar población numérica
geo_andalucia['POBLACION_MUNI'] = pd.to_numeric(geo_andalucia['POBLACION_MUNI'], errors='coerce')

# Filtrar > 10.000 habitantes
geo_andalucia_10k = geo_andalucia[geo_andalucia['POBLACION_MUNI'] > 2_500].copy()

# Construir dataframe final con las columnas pedidas
andaluces_10k_geo = geo_andalucia_10k[[
    'NOMBRE_ACTUAL',
    'PROVINCIA',
    'POBLACION_MUNI',
    'LATITUD_ETRS89',
    'LONGITUD_ETRS89'
]].copy()

# Renombrar columnas para que queden limpias
andaluces_10k_geo = andaluces_10k_geo.rename(columns={
    'NOMBRE_ACTUAL': 'municipio',
    'POBLACION_MUNI': 'poblacion',
    'LATITUD_ETRS89': 'latitud',
    'LONGITUD_ETRS89': 'longitud'
})

display(andaluces_10k_geo)
print(andaluces_10k_geo.shape)

,municipio,PROVINCIA,poblacion,latitud,longitud
281,Adra,Almería,25195,36.748683,-3.023303
284,Albox,Almería,12311,37.389678,-2.146835
289,Alhama de Almería,Almería,3817,36.957183,-2.569776
291,Almería,Almería,200578,36.838924,-2.464132
294,Antas,Almería,3417,37.244592,-1.917153
...,...,...,...,...,...
6172,Villaverde del Río,Sevilla,7700,37.587850,-5.873495
6173,El Viso del Alcor,Sevilla,19265,37.388975,-5.718532
6174,Cañada Rosal,Sevilla,3391,37.598739,-5.211294
6175,Isla Mayor,Sevilla,5741,37.133844,-6.162849


(406, 5)


In [8]:
ruta_destino = Path("..") / "data" / "processed" / "andaluces_2_5k.csv"
andaluces_10k_geo.to_csv(ruta_destino, index=False)